In [1]:
# %%
import pickle
import pandas as pd
from IPython.display import display, Markdown

# ==========================================
# ⏱️ 量化时光机：机构级每日战报
# ==========================================
TARGET_DATE = "2026-04-24"
report_file = f"Daily_Reports/Report_{TARGET_DATE}.pkl"

try:
    with open(report_file, 'rb') as f:
        snap = pickle.load(f)

    # 标题头
    display(Markdown(f"# 📊 宽客实战日报 | 交易日: {snap['date']}"))
    display(
        Markdown(f"### 👑 今日市场绝对主线 (King Factor): **`{snap['king_factor']}`**"))
    display(Markdown("---"))

    # 模块 1：OLS 市场风向归因
    r2_fac = snap['ols_report']['r2_factors'] * 100
    r2_ind = snap['ols_report']['r2_ind'] * 100
    display(Markdown(
        f"#### 📡 1. 市场微观结构探测 (解释力 - 因子: `{r2_fac:.2f}%` | 行业: `{r2_ind:.2f}%`)"))

    # ✨ 修复了 Pandas 的版本兼容警告：用 map 替代了 applymap
    style_ols = snap['ols_report']['stats_df'].style.map(
        lambda x: 'color: red; font-weight: bold' if isinstance(x, str) and '追捧 ★' in x else
                  ('color: green; font-weight: bold' if isinstance(x, str)
                   and '抛售 ★' in x else ''),
        subset=['风向状态']
    ).background_gradient(subset=['Beta系数'], cmap='coolwarm', vmin=-0.05, vmax=0.05)
    display(style_ols)

    # 模块 1.5：因子强度“英雄榜”
    display(
        Markdown(f"#### 🏆 1.5 因子全维度强度对比 (当前选股基准: `{snap['king_factor']}`)"))

    # 对统计表进行美化：Beta 用渐变色，T值绝对值大于 1.96 的加粗
    king_stats_styled = snap['king_stats'].style.background_gradient(
        subset=['Beta (强度)'], cmap='RdYlGn'
    ).map(
        lambda x: 'font-weight: bold; color: #FF4500' if isinstance(
            x, (int, float)) and abs(x) > 1.96 else '',
        subset=['T值 (显著性)']
    )

    display(king_stats_styled)

    # 计算次强因子与强度差距
    stats_df = snap['king_stats']
    if len(stats_df) > 1:
        top_1 = stats_df.iloc[0]
        top_2 = stats_df.iloc[1]
        gap = round(top_1['Beta (强度)'] - top_2['Beta (强度)'], 4)
        display(Markdown(
            f"> 💡 **复盘笔记**：今日最强因子 `{top_1['因子名称']}` 与次强因子 `{top_2['因子名称']}` 的强度差距为 **`{gap}`**。"))

    # 模块 2：强势板块与龙头画像
    display(Markdown("#### 🎯 2. 强势板块下钻与龙头画像"))
    # 横向展示领涨和领跌板块
    col1, col2 = snap['drill_report']['top_industries'], snap['drill_report']['bottom_industries']
    display(pd.concat([col1.set_index('板块名称'), col2.set_index(
        '板块名称')], axis=1, keys=['🔥 领涨板块', '❄️ 领跌板块']))

    display(Markdown("**【强势板块内部资金画像】**"))
    display(snap['drill_report']['portraits_df'].style.set_properties(
        **{'text-align': 'left'}))
    display(Markdown("---"))

    # 模块 3：明日交易底仓狙击池
    display(
        Markdown(f"#### 🔫 3. 明日开盘狙击池 (按核心因子 **`{snap['king_factor']}`** 降序排列)"))

    display_cols = ['code', 'name', snap['king_factor'],
                    'Momentum', 'Liquidity', 'Size']
    # 将更多你想看的因子加到表格里显示 (比如把 Amihud 也展示出来)：
    # display_cols = ['code', 'name', snap['king_factor'], 'Momentum', 'Liquidity', 'Size', 'Amihud']

    display_cols = list(dict.fromkeys(display_cols))

    # 原来是 .head(10)，改成 .head(20) 就能看前 20 名金股！
    top_picks = snap['top_picks'].head(20)[display_cols]

    display(top_picks.style.background_gradient(
        subset=[snap['king_factor']], cmap='YlOrRd'))


except FileNotFoundError:
    print(f"❌ 找不到 {TARGET_DATE} 的战报！")
# %%

# 📊 宽客实战日报 | 交易日: 2026-04-24

### 👑 今日市场绝对主线 (King Factor): **`Liquidity`**

---

#### 📡 1. 市场微观结构探测 (解释力 - 因子: `3.37%` | 行业: `17.18%`)

,因子 (Factor),Beta系数,T值 (显著性),风向状态
0,Momentum,11.891000,2.200000,追捧 ★
1,Short_Rev,7.977000,1.420000,追捧
2,Low_Vol,-9.806000,-1.720000,抛售
3,Liquidity,8.445000,1.560000,追捧
4,Size,9.029000,1.580000,追捧
5,Value_BP,1.411000,0.180000,追捧
6,Mom_Sharpe,12.707000,2.190000,追捧 ★
7,Vol_Price_Corr,-2.358000,-0.440000,抛售
8,Amihud,-8.978000,-1.520000,抛售


...

**【强势板块内部资金画像】**

,强势板块,代码,名称,预期超额涨幅,核心因子画像
0,Q84卫生,sz.300015,爱尔眼科,+16.17%,"低Mom_Sharpe(-1.1), 低Vol_Price_Corr(-1.1)"
1,Q84卫生,sh.600763,通策医疗,+4.90%,微盘股(-1.1)
2,Q84卫生,sz.002044,美年健康,+-2.26%,"低Momentum(-1.2), 低Mom_Sharpe(-1.2)"
3,M75科技推广和应用服务业,sz.003035,南网能源,+5.54%,低Low_Vol(-1.1)
4,B06煤炭开采和洗选业,sh.600188,兖矿能源,+8.84%,"高Size(+1.1), 低Mom_Sharpe(-1.0), 低Vol_Price_Corr(-1.2)"
5,B06煤炭开采和洗选业,sh.600348,华阳股份,+8.60%,中庸(随板块普涨)
6,B06煤炭开采和洗选业,sh.601699,潞安环能,+8.16%,"低Momentum(-1.1), 高Value_BP(+1.5), 低Mom_Sharpe(-1.0), 低Vol_Price_Corr(-1.2)"
7,B06煤炭开采和洗选业,sh.601918,新集能源,+8.13%,中庸(随板块普涨)
8,B06煤炭开采和洗选业,sh.600546,山煤国际,+7.80%,中庸(随板块普涨)
9,B06煤炭开采和洗选业,sh.600985,淮北矿业,+7.17%,高Value_BP(+1.5)


---

#### 🔫 3. 明日开盘狙击池 (按核心因子 **`Liquidity`** 降序排列)

,code,name,Liquidity,Momentum,Size
0,2460,赣锋锂业,3.176153,1.155876,0.851762
1,300394,天孚通信,2.170066,-0.440993,1.985279
2,2466,天齐锂业,1.944115,2.838365,0.784660
3,977,浪潮信息,1.908991,2.221135,0.853407
4,300274,阳光电源,1.830728,-2.161302,1.740492
5,300502,新易盛,1.524164,1.176069,2.608406
6,300567,精测电子,1.315120,0.040595,-0.548467
7,688347,华虹公司,1.311459,1.792705,0.060081
8,300014,亿纬锂能,1.192244,-0.047207,1.247349
9,688008,澜起科技,1.122798,1.337414,1.467319
